# 🧠 Taller de Cortex AI Functions en Snowflake
### ✅ Versión con Soluciones

---

### El share de este taller

Usamos **`SFC_SAMPLES.SAMPLE_DATA`**, el share público de Snowflake con el dataset **TPC-H**.


## 📦 Tablas en `TPCH_SF1`

| Tabla | Filas aprox. | Descripción |
|---|---|---|
| `ORDERS` | 1.5 M | Pedidos |
| `CUSTOMER` | 150 K | Clientes |
| `PART` | 200 K | Catálogo de productos |
| `SUPPLIER` | 10 K | Proveedores |
| `NATION` / `REGION` | 25 / 5 | Geografía |

---
## ⚙️ Configuración

Ejecuta esta celda primero. El rol activo necesita el database role `SNOWFLAKE.CORTEX_USER`.

In [ ]:
USE ROLE CURSO_DATA_ENGINEERING;
USE WAREHOUSE WH_CURSO_DATA_ENGINEERING;
USE DATABASE SNOWFLAKE_SAMPLE_DATA;
USE SCHEMA TPCH_SF1;

SHOW TABLES IN SCHEMA SNOWFLAKE_SAMPLE_DATA.TPCH_SF1;

---
## 🔍 Ejercicio 0 — Exploración de datos

Antes de usar funciones AI, familiarízate con los comentarios disponibles en `ORDERS`.

In [ ]:
SELECT O_ORDERKEY, O_ORDERSTATUS, O_TOTALPRICE, O_ORDERDATE, O_ORDERPRIORITY, O_COMMENT
FROM   ORDERS
WHERE  O_COMMENT IS NOT NULL AND LENGTH(O_COMMENT) > 30
LIMIT  10;

---
## 🏋️ Ejercicio 1 — `AI_SENTIMENT`

### Análisis de sentimiento por aspectos

`AI_SENTIMENT` va más allá del score numérico: devuelve **sentimiento por categorías** que tú defines (entrega, precio, calidad…), además del global.

Retorna un objeto JSON: `{ "categories": [ {"name":"overall","sentiment":"mixed"}, ... ] }`

**Tarea:**
1. Analiza 20 comentarios recientes de `ORDERS`.
2. Extrae sentimiento global y desglose por `delivery`, `price` y `quality`.
3. Filtra solo los comentarios con sentimiento global `negative` o `mixed`.

> 💡 **Sintaxis:** `AI_SENTIMENT(<texto>, ['aspecto1', 'aspecto2'])` — los aspectos son opcionales
>
> 💡 **Extraer del JSON:** `resultado:categories[0]:sentiment::STRING`
>
> El índice 0 siempre es `overall`. Los aspectos siguen en orden: índice 1, 2, 3…
>
> Valores posibles: `positive`, `negative`, `neutral`, `mixed`, `unknown`

In [ ]:
-- Ejercicio 1: AI_SENTIMENT con análisis por aspectos
WITH SENT_RAW AS (
    SELECT
        O_ORDERKEY,
        O_ORDERPRIORITY,
        O_COMMENT,
        AI_SENTIMENT(O_COMMENT, ['delivery', 'price', 'quality']) AS SENT_JSON
    FROM   ORDERS
    WHERE  O_COMMENT IS NOT NULL AND LENGTH(O_COMMENT) > 20
    ORDER  BY O_ORDERDATE DESC
    LIMIT  20
)
SELECT
    O_ORDERKEY,
    O_ORDERPRIORITY,
    O_COMMENT,
    SENT_JSON:categories[0]:sentiment::STRING   AS SENT_GLOBAL,
    SENT_JSON:categories[1]:sentiment::STRING   AS ASPECTO_DELIVERY,
    SENT_JSON:categories[2]:sentiment::STRING   AS ASPECTO_PRICE,
    SENT_JSON:categories[3]:sentiment::STRING   AS ASPECTO_QUALITY
FROM  SENT_RAW
WHERE SENT_JSON:categories[0]:sentiment::STRING IN ('negative', 'mixed')
ORDER BY O_ORDERKEY;

---
## 🏋️ Ejercicio 2 — `AI_TRANSLATE`

### Traducción con detección automática de idioma

`AI_TRANSLATE`  el **idioma origen es opcional** — puedes omitirlo y Snowflake lo detecta automáticamente.

**Tarea:**
1. Toma 10 comentarios de `PART` (campo `P_COMMENT`).
2. Tradúcelos al español sin especificar idioma origen.
3. Muestra nombre del producto, comentario original y traducción.

> 💡 **Sintaxis:** `AI_TRANSLATE(<texto>, '', <idioma_destino>')`
>
> Con idioma origen explícito: `AI_TRANSLATE(<texto>, '<destino>', '<origen>')`
>
> Códigos: `es` español · `en` inglés · `fr` francés · `de` alemán · `pt` portugués · `ja` japonés

In [ ]:
-- Ejercicio 2: AI_TRANSLATE con detección automática de idioma
SELECT
    P_PARTKEY,
    P_NAME,
    P_MFGR,
    P_COMMENT                         AS COMENTARIO_ORIGINAL,
    AI_TRANSLATE(P_COMMENT, '', 'es')     AS COMENTARIO_EN_ESPANOL
FROM  PART
WHERE P_COMMENT IS NOT NULL AND LENGTH(P_COMMENT) BETWEEN 20 AND 120
LIMIT 10;

---
## 🏋️ Ejercicio 3 — `AI_SUMMARIZE_AGG`

### Resumen agregado sin límite de contexto

`AI_SUMMARIZE_AGG` es una **función de agregación nativa** — funciona igual que `COUNT()` o `SUM()`, pero sobre texto. No tiene límite de ventana de contexto.

**Tarea:**
1. Identifica los 5 clientes con mayor gasto total.
2. Para cada uno, usa `AI_SUMMARIZE_AGG` en un `GROUP BY` para resumir todos sus comentarios de pedido.
3. Muestra cliente, gasto y el resumen generado.

> 💡 **Sintaxis:** `AI_SUMMARIZE_AGG(<columna_texto>)` — úsalo como cualquier función de agregación
>
> 💡 Pista: usa una CTE para identificar los top 5 y luego el GROUP BY con la función

In [ ]:
-- Ejercicio 3: AI_SUMMARIZE_AGG como función de agregación nativa
WITH TOP5 AS (
    SELECT O_CUSTKEY, SUM(O_TOTALPRICE) AS GASTO_TOTAL
    FROM   ORDERS
    GROUP  BY O_CUSTKEY
    ORDER  BY GASTO_TOTAL DESC
    LIMIT  5
)
SELECT
    C.C_NAME,
    ROUND(T.GASTO_TOTAL, 2)        AS GASTO_TOTAL_USD,
    AI_SUMMARIZE_AGG(O.O_COMMENT)  AS RESUMEN_ACTIVIDAD
FROM   TOP5      T
JOIN   CUSTOMER  C ON C.C_CUSTKEY = T.O_CUSTKEY
JOIN   ORDERS    O ON O.O_CUSTKEY = T.O_CUSTKEY
WHERE  O.O_COMMENT IS NOT NULL
GROUP  BY C.C_NAME, T.GASTO_TOTAL
ORDER  BY T.GASTO_TOTAL DESC;

---
## 🏋️ Ejercicio 4 — `AI_CLASSIFY`

### Clasificación con ejemplos few-shot y multi-label

`AI_CLASSIFY`
- **`examples`** por categoría → mejora precisión con ejemplos concretos (*few-shot*)
- **`output_mode: 'multi'`** → un texto puede recibir varias categorías a la vez

**Tarea:**
1. Toma 15 comentarios de `SUPPLIER`.
2. Clasifícalos en estas categorías de negocio: `logistica`, `calidad`, `precio`, `servicio`, `otro`.
3. Incluye `description` y al menos un `example` por categoría.
4. Extrae label y score de confianza del JSON resultado.

> 💡 **Sintaxis:**
> ```sql
> AI_CLASSIFY(
>   <texto>,
>   [ {'label':'cat', 'description':'...', 'examples':['ej1','ej2']}, ... ],
>   {'output_mode': 'single'}
> )
> ```
> Devuelve JSON: `{"label":"...", "score":0.95}` en modo single
>
> 💡 Pista: usa una CTE para guardar el JSON y extrae `resultado:label::STRING` en el SELECT exterior

In [ ]:
-- Ejercicio 4: AI_CLASSIFY con descripción y ejemplos few-shot
WITH CLASSIFIED AS (
    SELECT
        S_SUPPKEY, S_NAME, S_COMMENT,
        AI_CLASSIFY(
            S_COMMENT,
            [
                {'label':'logistica',  'description':'Problemas de entrega, envío o transporte',
                 'examples':['shipment delayed', 'delivery took too long']},
                {'label':'calidad',    'description':'Calidad del producto, defectos o estado del material',
                 'examples':['parts were defective', 'excellent build quality']},
                {'label':'precio',     'description':'Costes, tarifas, facturas o negociación de precios',
                 'examples':['prices are too high', 'competitive rates']},
                {'label':'servicio',   'description':'Atención al cliente, soporte o comunicación',
                 'examples':['support team was helpful', 'no response to complaints']},
                {'label':'otro',       'description':'No encaja en ninguna categoría anterior'}
            ],
            {'output_mode': 'single'}
        ) AS RESULT_JSON
    FROM  SUPPLIER
    WHERE S_COMMENT IS NOT NULL AND LENGTH(S_COMMENT) > 15
    LIMIT 15
)
SELECT
    S_SUPPKEY, S_NAME, S_COMMENT,
    RESULT_JSON:labels[0]::STRING             AS CATEGORIA,
FROM  CLASSIFIED
ORDER BY CONFIANZA DESC;

---
## 🏋️ Ejercicio 5 — `AI_COMPLETE` + `PROMPT()`

### Generación de texto con modelos de última generación

`AI_COMPLETE` 
- Modelos actualizados: `claude-sonnet-4-5`, `gpt-5`, `llama4-maverick`, `gemini-2-5-flash`…
- El helper **`PROMPT('texto con {0} placeholders', col1, col2)`** construye prompts dinámicos sin concatenar strings.
- Parámetros opcionales: `{'temperature': 0.3, 'max_tokens': 200}`

**Tarea:**
1. Selecciona los 5 pedidos urgentes de mayor valor (join `ORDERS` + `CUSTOMER`).
2. Usa `AI_COMPLETE` + `PROMPT()` incluyendo cliente, fecha, importe, estado y comentario.
3. Genera una nota ejecutiva de seguimiento en español, máx. 50 palabras por pedido.

> 💡 **Sintaxis:** `AI_COMPLETE('<modelo>', PROMPT('texto {0}', columna))`
>
> Con parámetros: `AI_COMPLETE('<modelo>', prompt_obj, {'temperature': 0.3, 'max_tokens': 120})`
>
> 💡 Pista: CTE para preparar datos → AI_COMPLETE en el SELECT exterior

In [ ]:
-- Ejercicio 5: AI_COMPLETE con PROMPT() y parámetros
WITH URGENTES AS (
    SELECT
        O.O_ORDERKEY,
        C.C_NAME                                     AS CLIENTE,
        O.O_ORDERDATE                                AS FECHA,
        O.O_TOTALPRICE                               AS IMPORTE,
        O.O_ORDERSTATUS                              AS ESTADO,
        COALESCE(O.O_COMMENT, 'sin comentario')      AS COMENTARIO
    FROM  ORDERS   O
    JOIN  CUSTOMER C ON C.C_CUSTKEY = O.O_CUSTKEY
    WHERE O.O_ORDERPRIORITY = '1-URGENT'
    ORDER BY O.O_TOTALPRICE DESC
    LIMIT 5
)
SELECT
    O_ORDERKEY,
    CLIENTE,
    FECHA,
    ROUND(IMPORTE, 2)   AS IMPORTE_USD,
    ESTADO,
    AI_COMPLETE(
        'claude-sonnet-4-5',
        PROMPT(
            'Eres un asistente de ventas. Escribe una nota de seguimiento ejecutiva en español, 
            máximo 50 palabras, para este pedido urgente. 
            Cliente: {0}. Fecha: {1}. Importe: ${2}. Estado: {3}. Nota original: {4}. 
            Responde SOLO con la nota.',
            CLIENTE, FECHA::STRING, ROUND(IMPORTE,2)::STRING, ESTADO, COMENTARIO
        ),
        {'temperature': 0.3, 'max_tokens': 120}
    )                   AS NOTA_EJECUTIVA
FROM URGENTES;

---
## 🏋️ Ejercicio 6 — `AI_FILTER`

### Filtrado semántico en cláusula WHERE

`AI_FILTER`  devuelve `TRUE`/`FALSE` evaluando una condición en lenguaje natural sobre texto. Úsala en `SELECT`, `WHERE` o `JOIN ... ON`.

Reemplaza búsquedas con `ILIKE '%palabra%'` que no capturan sinónimos ni contexto implícito.

**Tarea:**
1. Filtra proveedores (`SUPPLIER` + join con `NATION`) cuyo comentario mencione **problemas de retraso o entrega pendiente**.
2. En el SELECT muestra también el resultado de `ILIKE '%delay%' OR ILIKE '%late%'` para comparar.
3. Aplica `AI_FILTER` directamente en el `WHERE`.

> 💡 **Sintaxis:** `AI_FILTER('<condición en lenguaje natural>')`
>
> Usable en: `WHERE AI_FILTER(...)` · `SELECT AI_FILTER(...) AS flag` · `JOIN ... ON AI_FILTER(...)`
>
> 💡 No necesita CTE — puedes llamarla dos veces (en SELECT y WHERE) con la misma condición

In [ ]:
-- Ejercicio 6: AI_FILTER — filtrado semántico vs búsqueda por palabras clave
SELECT
    S.S_SUPPKEY,
    S.S_NAME,
    N.N_NAME                                                         AS PAIS,
    S.S_COMMENT,
    -- ¿El ILIKE clásico también lo captura?
    (S.S_COMMENT ILIKE '%delay%' OR S.S_COMMENT ILIKE '%late%')      AS ILIKE_MATCH
FROM  SUPPLIER S
JOIN  NATION   N ON N.N_NATIONKEY = S.S_NATIONKEY
WHERE S.S_COMMENT IS NOT NULL
  AND LENGTH(S.S_COMMENT) > 15
  -- Filtro semántico aplicado directamente en WHERE
  AND AI_FILTER(
        CONCAT('The following text mentions delivery problems, delays, or shipments that are pending or overdue:',
        S.S_COMMENT)
      )
LIMIT 20;

---
## 🏋️ Ejercicio 7 — `AI_EXTRACT`

### Extracción multi-campo en una sola llamada

`AI_EXTRACT`  La mejora clave: soporta **múltiples preguntas en una llamada** devolviendo un JSON con todas las respuestas.

**Tarea:**
1. Usa comentarios de `SUPPLIER` (join con `NATION`).
2. En **una sola llamada** a `AI_EXTRACT`, extrae tres campos: el problema, el producto afectado y si hay urgencia.
3. Filtra las filas donde el campo `problema` no esté vacío.

> 💡 **Sintaxis multi-campo:**
> ```sql
> AI_EXTRACT(
>   <texto>,
>   {'campo1': '<pregunta1>', 'campo2': '<pregunta2>', ...}
> )
> ```
> Devuelve un OBJECT JSON — extrae con `resultado:response:campo1::STRING`
>
> 💡 Pista: CTE para guardar el JSON → SELECT exterior para parsear y filtrar

In [ ]:
-- Ejercicio 7: AI_EXTRACT — múltiples campos en una sola llamada
WITH EXT AS (
    SELECT
        S.S_SUPPKEY,
        S.S_NAME,
        N.N_NAME       AS PAIS,
        S.S_COMMENT    AS CONTEXTO,
        AI_EXTRACT(
            S.S_COMMENT,
            {
                'problema'  : 'What specific issue or complaint is described?',
                'producto'  : 'What product or part is affected, if mentioned?',
                'urgencia'  : 'Is there an implicit sense of urgency or escalation?'
            }
        ) AS EXTRACCION
    FROM  SUPPLIER S
    JOIN  NATION   N ON N.N_NATIONKEY = S.S_NATIONKEY
    WHERE S.S_COMMENT IS NOT NULL AND LENGTH(S.S_COMMENT) > 20
    LIMIT 30
)
SELECT
    S_SUPPKEY, S_NAME, PAIS, CONTEXTO,
    EXTRACCION:problema::STRING    AS PROBLEMA_DETECTADO,
    EXTRACCION:producto::STRING    AS PRODUCTO_AFECTADO,
    EXTRACCION:urgencia::STRING    AS HAY_URGENCIA
FROM  EXT
WHERE EXTRACCION:problema::STRING IS NOT NULL
  AND EXTRACCION:problema::STRING != ''
ORDER BY S_NAME;

---
## 🏋️ Ejercicio 8 — `AI_AGG`

### Agregación con instrucción personalizada

`AI_AGG`  Se diferencia de `AI_SUMMARIZE_AGG` en que tú defines exactamente qué insight quieres extraer del grupo — como darle una instrucción específica a un analista.

**Tarea:**
Para cada prioridad de pedido (`O_ORDERPRIORITY`), usa `AI_AGG` con un prompt que devuelva:
- Los 2-3 temas recurrentes
- El tono predominante (positivo/negativo/neutro)
- Una recomendación de acción en una frase

> 💡 **Sintaxis:** `AI_AGG(<columna_texto>, '<instrucción_de_análisis>')`
>
> La instrucción puede pedir formato estructurado. Sin límite de filas por grupo.
>
> 💡 Pista: usa `QUALIFY ROW_NUMBER() OVER (PARTITION BY O_ORDERPRIORITY ORDER BY RANDOM()) <= 50` para muestrear y controlar el coste

In [ ]:
-- Ejercicio 8: AI_AGG con instrucción personalizada por prioridad de pedido
WITH MUESTRA AS (
    -- 50 comentarios por grupo para controlar coste
    SELECT O_ORDERPRIORITY, O_COMMENT
    FROM   ORDERS
    WHERE  O_COMMENT IS NOT NULL AND O_ORDERDATE >= '1997-01-01'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY O_ORDERPRIORITY ORDER BY RANDOM()) <= 50
)
SELECT
    O_ORDERPRIORITY,
    COUNT(*)     AS COMENTARIOS_ANALIZADOS,
    AI_AGG(
        O_COMMENT,
        'Analiza estos comentarios de pedidos y genera un resumen ejecutivo en español con: 
        1) Los 2-3 temas más recurrentes. 
        2) Si el tono predominante es positivo, negativo o neutro. 
        3) Una recomendación de acción en una frase. 
        Máximo 80 palabras en total.'
    )            AS ANALISIS_EJECUTIVO
FROM   MUESTRA
GROUP  BY O_ORDERPRIORITY
ORDER  BY O_ORDERPRIORITY;

---
## 📊 Resumen de funciones `AI_*` del taller

| Función | Tipo | Reemplaza a | Novedad clave |
|---|---|---|---|
| `AI_SENTIMENT` | Task-specific | `SNOWFLAKE.CORTEX.SENTIMENT` | Análisis por aspectos, multilingüe |
| `AI_TRANSLATE` | Task-specific | `SNOWFLAKE.CORTEX.TRANSLATE` | Detección automática de idioma origen |
| `AI_SUMMARIZE_AGG` | Agregación | `SNOWFLAKE.CORTEX.SUMMARIZE` | Función agregada nativa, sin límite contexto |
| `AI_CLASSIFY` | Task-specific | `SNOWFLAKE.CORTEX.CLASSIFY_TEXT` | Few-shot examples, multi-label, imágenes |
| `AI_COMPLETE` | Generativa | `SNOWFLAKE.CORTEX.COMPLETE` | Modelos actualizados, helper `PROMPT()` |
| `AI_FILTER` | Task-specific | *(nueva)* | Filtrado semántico booleano en WHERE/JOIN |
| `AI_EXTRACT` | Task-specific | `SNOWFLAKE.CORTEX.EXTRACT_ANSWER` | Multi-campo en una llamada |
| `AI_AGG` | Agregación | *(nueva)* | Insight personalizado sin límite de contexto |

### 🔗 Documentación oficial
- [Cortex AI Functions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)
- [AI_SENTIMENT con aspectos](https://docs.snowflake.com/en/sql-reference/functions/ai_sentiment)
- [AI_COMPLETE + modelos disponibles](https://docs.snowflake.com/en/sql-reference/functions/ai_complete)
- [AI_FILTER](https://docs.snowflake.com/en/sql-reference/functions/ai_filter)
- [Data Sharing](https://docs.snowflake.com/en/user-guide/data-sharing-intro)

> ✅ **Controlar costes:** Usa `AI_COUNT_TOKENS('ai_complete', texto, 'modelo')` para estimar tokens antes de queries masivas. Añade `QUALIFY ROW_NUMBER() ... <= N` para muestrear por grupo.